# Pixels to Predictions — Inference Notebook

**Task:** SmolVLM-500M-Instruct + LoRA adapter + retrieval overlay. Loads a trained adapter and writes `submission.csv` for the Kaggle test set, then applies the retrieval overlay that produced our best public-LB result of 0.861.

**Setup before running:** 
1. Download the trained iter4-vd adapter from https://drive.google.com/file/d/1H9PaPqCkqgfRtIRiNhHy4e5lf58VGJ63/view?usp=sharing and unzip into `./adapter_best/` (relative to the repo root).
2. Download the competition data via `kagglehub.competition_download('pixels-to-predictions')` and symlink into `./data/` per `DATA.md`.
3. Set `ADAPTER_PATH` and `DATA_DIR` below if your paths differ from the defaults.

**AI tooling disclosure:** Claude Code (Opus) was used as a coding/debugging assistant during development. Experimental design decisions were the author's. See the report `\section*{AI Tooling Disclosure}` for details.

**Reproducibility:** all random seeds are fixed via `src.run._seed_all`. Default inference seed is `42`.

In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path('..').resolve()  # adjust if running from a different directory
DATA_DIR = Path(os.environ.get('DATA_DIR', REPO_ROOT / 'data'))
ADAPTER_PATH = Path(os.environ.get('ADAPTER_PATH', REPO_ROOT / 'adapter_best'))
HF_CACHE = Path(os.environ.get('HF_HOME', REPO_ROOT / 'hf_cache'))
OUT_DIR = REPO_ROOT / 'runs' / 'infer-iter4-vd'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'REPO_ROOT={REPO_ROOT}')
print(f'DATA_DIR={DATA_DIR}')
print(f'ADAPTER_PATH={ADAPTER_PATH}')
print(f'HF_CACHE={HF_CACHE}')
print(f'OUT_DIR={OUT_DIR}')

In [ ]:
# Seed everything.
import sys
sys.path.insert(0, str(REPO_ROOT))
from src.run import _seed_all
_seed_all(42)

In [ ]:
# Configure HF cache and load model + adapter.
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE / 'transformers')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# First run downloads from HF; subsequent runs can be offline. Toggle the next two lines to force offline:
# os.environ['HF_HUB_OFFLINE'] = '1'
# os.environ['TRANSFORMERS_OFFLINE'] = '1'

import torch
from peft import PeftModel
from transformers import AutoModelForVision2Seq, AutoProcessor

MODEL_ID = 'HuggingFaceTB/SmolVLM-500M-Instruct'
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

base = AutoModelForVision2Seq.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base, str(ADAPTER_PATH), is_trainable=False)
if torch.cuda.is_available():
    model = model.to('cuda')
model.eval()
print('model + adapter loaded')

In [ ]:
# Run inference on val and test. Dumps val_scores.json so the retrieval overlay can use it.
import json
from src.data import load_split
from src.zero_shot import predict_zero_shot
from src.submission import write_submission

val_df = load_split(DATA_DIR / 'val.csv', labeled=True)
test_df = load_split(DATA_DIR / 'test.csv', labeled=False)

val_preds, val_scores = predict_zero_shot(
    val_df, data_dir=DATA_DIR, processor=processor, model=model,
    img_size=224, batch_size=32, num_workers=4, return_scores=True,
)
test_preds, test_scores = predict_zero_shot(
    test_df, data_dir=DATA_DIR, processor=processor, model=model,
    img_size=224, batch_size=32, num_workers=4, return_scores=True,
)

# Write submission.csv
pred_map = dict(zip(test_df['id'], test_preds))
submission_path = write_submission(pred_map, test_df, OUT_DIR / 'submission.csv')
print(f'wrote {submission_path}')

# Write val_scores.json (needed by the retrieval overlay).
val_records = []
for i in range(len(val_df)):
    row = val_df.iloc[i]
    val_records.append({
        'id': str(row['id']),
        'num_choices': int(row['num_choices']),
        'scores': [(s if s != float('-inf') else None) for s in val_scores[i]],
        'pred': int(val_preds[i]),
        'gold': int(row['answer']),
    })
(OUT_DIR / 'val_scores.json').write_text(json.dumps({'records': val_records}))
print(f'wrote {OUT_DIR / "val_scores.json"}')

## Retrieval overlay (the post-processor that produced 0.861)

Applies pHash + question-similarity + choice-match retrieval over training neighbors. Builds and caches the pHash index at `data/phash_cache/{train,val,test}_phash.json` on first run (~30–90 s).

In [ ]:
import subprocess
result = subprocess.run([
    'python', str(REPO_ROOT / 'scripts' / 'retrieval_overlay.py'),
    '--data-dir', str(DATA_DIR),
    '--base-submission', str(OUT_DIR / 'submission.csv'),
    '--base-val-scores', str(OUT_DIR / 'val_scores.json'),
    '--hamming-thresh', '4',
    '--qsim-thresh', '0.85',
    '--require-choice-match',
    '--out', str(REPO_ROOT / 'runs' / 'retrieval-vd-h4q085'),
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
else:
    print('\nfinal submission: runs/retrieval-vd-h4q085/submission_retrieval.csv')